In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import zscore

In [3]:
df = pd.read_csv('../data/nigeria.csv', encoding='latin1', skiprows=[1])
df['Country'] = 'Nigeria'
print(df.shape)
df.head()

(4107, 13)


,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,T2M_RANGE,PRECTOTCORR,RH2M,WS2M,WS2M_MAX,PS,QV2M,Country
0,2015,2,26.16,29.41,22.87,6.54,0.0,73.23,1.42,1.95,100.94,15.37,Nigeria
1,2015,3,25.66,29.02,22.63,6.39,0.0,78.71,1.69,2.33,101.06,15.98,Nigeria
2,2015,4,24.11,27.27,19.92,7.35,0.0,63.66,2.15,3.80,101.09,11.65,Nigeria
3,2015,5,23.40,27.28,18.18,9.10,0.0,59.45,1.88,3.48,101.03,10.40,Nigeria
4,2015,6,22.66,25.77,18.03,7.74,0.0,62.57,1.37,2.10,101.00,10.61,Nigeria


In [4]:
df["Date"] = pd.to_datetime(df["YEAR"] * 1000 + df["DOY"], format="%Y%j")
df['MONTH'] = df['Date'].dt.month
df.tail()

,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,T2M_RANGE,PRECTOTCORR,RH2M,WS2M,WS2M_MAX,PS,QV2M,Country,Date,MONTH
4102,2026,86,29.05,32.43,26.69,5.74,2.82,76.60,2.35,3.39,100.58,19.10,Nigeria,2026-03-27,3
4103,2026,87,28.72,31.98,27.14,4.84,5.19,79.61,2.55,3.17,100.64,19.49,Nigeria,2026-03-28,3
4104,2026,88,27.72,29.53,26.21,3.32,1.43,82.83,1.10,1.78,100.61,19.22,Nigeria,2026-03-29,3
4105,2026,89,28.42,31.17,26.36,4.81,0.85,77.73,2.30,3.40,100.53,18.73,Nigeria,2026-03-30,3
4106,2026,90,28.40,31.73,26.32,5.41,5.33,78.79,1.76,2.80,100.59,18.94,Nigeria,2026-03-31,3


In [5]:
# Check for missing values and duplicates(-999 is NASA's sentinel value for missing or out-of-range data.)
df = df.replace(-999, np.nan)
print("Number of duplicate rows found:", df.duplicated().sum())
df = df.drop_duplicates()

Number of duplicate rows found: 0


No duplicate rows were found and removed using drop_duplicates().
Duplicates were exact row-level repeats across all columns (no partial column-specific duplicates detected).

In [6]:
df.describe()

,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,T2M_RANGE,PRECTOTCORR,RH2M,WS2M,WS2M_MAX,PS,QV2M,Date,MONTH
count,4107.000000,4107.000000,4107.000000,4107.000000,4107.000000,4107.000000,4107.000000,4107.000000,4107.000000,4107.000000,4107.000000,4107.000000,4107,4107.000000
mean,2020.132700,180.164841,26.657275,28.914585,24.887149,4.027436,4.214940,85.241174,2.217253,2.903406,100.827197,18.559771,2020-08-16 00:00:00,6.424884
min,2015.000000,1.000000,21.120000,25.260000,15.170000,1.160000,0.000000,54.400000,0.740000,1.290000,100.380000,9.430000,2015-01-02 00:00:00,1.000000
25%,2017.000000,86.000000,25.720000,27.920000,24.105000,3.090000,0.330000,83.935000,1.770000,2.370000,100.710000,17.970000,2017-10-24 12:00:00,3.000000
50%,2020.000000,179.000000,26.820000,28.990000,25.100000,3.770000,1.840000,86.350000,2.200000,2.810000,100.820000,18.840000,2020-08-16 00:00:00,6.000000
75%,2023.000000,272.000000,27.540000,29.910000,25.860000,4.600000,5.200000,88.500000,2.630000,3.390000,100.950000,19.570000,2023-06-08 12:00:00,9.000000
max,2026.000000,366.000000,29.290000,32.880000,27.790000,11.730000,166.100000,93.790000,4.780000,6.000000,101.350000,21.740000,2026-03-31 00:00:00,12.000000
std,3.248315,106.270943,1.123251,1.294492,1.396200,1.398469,7.267329,5.440221,0.587213,0.696955,0.165341,1.644513,NaN,3.476439


### Updated Descriptive Statistics After Cleaning

The cleaned dataset contains 4,107 observations.

Temperature variables (T2M, T2M_MAX, T2M_MIN) show relatively low variability, with mean values around 26.66°C, 28.91°C, and 24.89°C respectively. The small difference between mean and median suggests a fairly symmetric distribution.

Relative humidity (RH2M) is consistently high, with an average of approximately 85.24%, indicating a generally humid climate. The relatively low standard deviation suggests stable humidity conditions.

Precipitation (PRECTOTCORR) remains highly variable, with a mean of 4.21 but a maximum of 166.1. This indicates occasional extreme rainfall events, while most values remain relatively low.

Wind speed (WS2M) and maximum wind speed (WS2M_MAX) show moderate variability, with generally low to متوسط wind conditions.

Surface pressure (PS) remains very stable across the dataset, with minimal variation, which is typical for atmospheric pressure.

Specific humidity (QV2M) shows moderate variation, reflecting changes in atmospheric moisture content.

Overall, the dataset appears clean and consistent, with reduced noise and improved reliability for further analysis. The presence of extreme precipitation values suggests potential outliers or significant weather events that may require further investigation.

In [7]:
df.isna().sum() 
missing_percent = (df.isna().sum() / len(df)) * 100
print(missing_percent)

YEAR           0.0
DOY            0.0
T2M            0.0
T2M_MAX        0.0
T2M_MIN        0.0
T2M_RANGE      0.0
PRECTOTCORR    0.0
RH2M           0.0
WS2M           0.0
WS2M_MAX       0.0
PS             0.0
QV2M           0.0
Country        0.0
Date           0.0
MONTH          0.0
dtype: float64


In [8]:
cols = ["T2M", "T2M_MAX", "T2M_MIN", "PRECTOTCORR", "RH2M", "WS2M", "WS2M_MAX"]

z_scores = df[cols].apply(zscore)
outliers = (np.abs(z_scores) > 3)
outlier_rows = outliers.any(axis=1).sum()
print("Number of rows with extreme values (|Z| > 3):", outlier_rows)

Number of rows with extreme values (|Z| > 3): 225



### Z-Score Outlier Analysis
Rows with absolute Z-scores greater than 3 were flagged as potential outliers, indicating unusually extreme weather conditions compared to the dataset distribution.

The analysis identified 225 rows containing at least one extreme value. These outliers may correspond to rare weather events such as heatwaves, heavy rainfall, or unusually strong winds.

Such extreme values are important in climate analysis and should not be automatically removed, as they may represent meaningful environmental phenomena rather than data errors.

In [9]:
# Forward-fill weather-related columns
weather_cols = ["T2M", "T2M_MAX", "T2M_MIN", "PRECTOTCORR", "RH2M", "WS2M", "WS2M_MAX"]
df[weather_cols] = df[weather_cols].ffill()
threshold = int(0.7 * df.shape[1])  # keep rows with at least 70% non-null values
df = df.dropna(thresh=threshold)


### Missing Value Handling

Missing values were handled using a combination of forward-fill imputation and row removal.

Forward-fill was applied to key weather variables (temperature, precipitation, humidity, and wind speed) to preserve temporal continuity in the dataset. This method assumes that short gaps in measurements can be reasonably approximated using the most recent valid observation.

Additionally, rows with more than 30% missing values were removed to ensure data quality and reliability. This prevents heavily incomplete records from skewing analysis.

In this dataset, no missing values remained after initial preprocessing, so these steps served as a precautionary measure to ensure robustness.

In [10]:
df.to_csv("../data/nigeria_clean.csv", index=False)